In [0]:
path_design_campaign = "/Volumes/sandbox/karthik/motif_scaffolding/alpha2_beta1_double_globular_trial3"
path_boltz_folder = f"{path_design_campaign}/rfdiffusion/rfd_af2_passed_boltz_input"

In [0]:
import pandas as pd
import numpy as np
import re
import seaborn as sns
import matplotlib.pyplot as plt


list_df_chunks = []
for chunk_id in range(0, 3):
    path_boltz_chunk = f"{path_boltz_folder}/boltz_holo_rfd_af2_passed_chunk_{chunk_id}.csv"
    df_boltz_chunk = pd.read_csv(path_boltz_chunk)
    list_df_chunks.append(df_boltz_chunk)
df_boltz_designs = pd.concat(list_df_chunks, axis = 0)
df_boltz_designs.rename(columns = {'ptm' : 'ptm_boltz'}, inplace= True)
df_boltz_designs

In [0]:
df_boltz_most_confident = df_boltz_designs[df_boltz_designs['model_id'] == 0]
df_boltz_most_confident

In [0]:
figsize = (8, 6)
data_col = 'rmsd_holo_binder_rfd_boltz'
fig, ax = plt.subplots(figsize = figsize)
sns.histplot(data = df_boltz_most_confident, x = data_col, bins = 100, ax = ax)
ax.set_xlabel(data_col)
ax.set_ylabel("Number of designs")
ax.set_title(f"Distribution of {data_col} for {df_boltz_most_confident.shape[0]} designs");

In [0]:
agg_funcs = ['median', 'mean', 'std']
df_boltz_groupedby = df_boltz_designs.groupby('design_name').agg({'ptm_boltz' : agg_funcs, 'ligand_iptm' : agg_funcs, 'protein_iptm' : agg_funcs, 'complex_plddt' : agg_funcs, 'complex_iplddt' : agg_funcs, 'complex_pde' : agg_funcs, 'rmsd_holo_binder_rfd_boltz' : agg_funcs})
df_boltz_groupedby.sort_values(by = ('rmsd_holo_binder_rfd_boltz', 'median'), ascending = True, inplace = True)
df_boltz_groupedby.reset_index(inplace = True)
df_boltz_groupedby

In [0]:
rmsd_threshold = 4
df_top_designs = df_boltz_groupedby[df_boltz_groupedby['rmsd_holo_binder_rfd_boltz']['median'] < rmsd_threshold]
df_top_designs.columns = ['_'.join(col).strip('_') for col in df_top_designs.columns]
print(f"Number of designs with RMSD < {rmsd_threshold}: {df_top_designs.shape[0]}")
df_top_designs

In [0]:
data_col = 'complex_plddt'
df_top_designs[f"{data_col}_mean"].describe()

In [0]:
df_boltz_top_predictions = pd.merge(df_boltz_designs, df_top_designs, on = 'design_name', how = 'inner')
df_boltz_top_predictions.sort_values(by = ['rmsd_holo_binder_rfd_boltz_median', 'model_id'], ascending = [True, True], inplace = True)
df_boltz_top_predictions.reset_index(drop = True, inplace = True)
df_boltz_top_predictions

In [0]:
df_boltz_top_predictions['design_name'].iloc[0]

In [0]:
import py2Dmol
import sys
import random
sys.path.append('/Workspace/Users/karthik.raj@bio-techne.com/ScoringPhysics')
from StrucTools import *
df_boltz_top_predictions_model0 = df_boltz_top_predictions[df_boltz_top_predictions['model_id'] == 0]
random_design = random.randint(0, df_boltz_top_predictions_model0.shape[0]) 
print(f"Index of random design: {random_design}")
print(f"Design name: {df_boltz_top_predictions_model0['design_name'].iloc[random_design]}")
print(f"RMSD: {df_boltz_top_predictions_model0['rmsd_holo_binder_rfd_boltz'].iloc[random_design]}")
print(f"Confidence Score: {df_boltz_top_predictions_model0['confidence_score'].iloc[random_design]}")
path_structure = df_boltz_top_predictions_model0['path_structure'].iloc[random_design]
visualize_structure(path_structure)

### Merging Back to AF2 Outputs

In [0]:
path_design_campaign

In [0]:
list_df_af2 = []
for chunk_id in range(0, 3):
    path_af2_chunk = f"{path_design_campaign}/rfdiffusion/rfd_af2_passed_boltz_input/rfd_af2_passed_chunk_{chunk_id}.csv"
    df_af2_chunk = pd.read_csv(path_af2_chunk, index_col= 0)
    list_df_af2.append(df_af2_chunk)
df_af2 = pd.concat(list_df_af2, axis = 0)
df_af2

In [0]:
df_boltz_holo_af2 = pd.merge(left = df_boltz_top_predictions, right = df_af2, on = 'design_name', how = 'inner')
df_boltz_holo_af2.rename(columns = {'path_structure' : 'path_holo_boltz_structure', 
                                    'design_pdb_path' : 'path_holo_af2_structure'}, inplace= True)
df_boltz_holo_af2

In [0]:
df_boltz_holo_af2_designs = df_boltz_holo_af2[df_boltz_holo_af2['model_id'] == 0].reset_index(drop = True)
df_boltz_holo_af2_designs

In [0]:
path_design_campaign

In [0]:
path_save_folder = f"{path_design_campaign}/rfdiffusion/final_metrics"
df_boltz_holo_af2_designs.to_csv(f"{path_save_folder}/boltz_af2_holo_passed_median_rmsd_less_{rmsd_threshold}.csv")
#df_boltz_holo_af2.to_csv(f"{path_save_folder}/boltz_af2_holo_passed_median_rmsd_less_{rmsd_threshold}_all_models.csv")

### Figuring out Yamls consistently producing/sampling top designs

In [0]:
df_boltz_top_predictions_model0['design_name'].values

In [0]:
df_boltz_top_design_names = df_boltz_top_predictions_model0['design_name'].str.extract(r"((\w+)_(\d+)-(\d+)_(\w+)_(\d+)-(\d+))", expand = True)
df_boltz_top_design_names.rename(columns = {0: 'design_yaml', 1 : 'design_num'}, inplace = True)
rmsd_lessthan_5_yaml_value_counts = df_boltz_top_design_names['design_yaml'].value_counts()
rmsd_lessthan_5_yaml_value_counts

In [0]:
top_yamls = list(rmsd_lessthan_5_yaml_value_counts.index)
print(f"Number of unique yamls: {len(top_yamls)}")
sorted(top_yamls)

In [0]:
os.

In [0]:
import os
yamls = [x for x in os.listdir("/Volumes/sandbox/karthik/motif_scaffolding/alpha2_beta1_double_globular_trial2/rfdiffusion/yaml/") if "base" not in x]
sorted(yamls) == sorted(top_yamls)

In [0]:
yamls * 2

In [0]:
for yaml in top_yamls:
    print(yaml)
    print(f"{yaml}.yaml" in yamls)